In [50]:
import sys
sys.path.insert(0, '..')

In [51]:
from data.dataset import load_raw_data, extract_labeled_patterned_data

df = load_raw_data()

print(df.shape)
print(df.columns.tolist())

(811457, 6)
['waferMap', 'dieSize', 'lotName', 'waferIndex', 'trianTestLabel', 'failureType']


In [63]:
import numpy as np

df_with_label, df_with_pattern, df_without_pattern = extract_labeled_patterned_data(df)

CLASS_COUNTS = {
  "Center": 3,
    "Donut": 2,
    "Edge-Loc": 4,
    "Edge-Ring": 2,
    "Loc": 3,
    "Near-full": 1,
    "Random": 2,
    "Scratch": 3,
}

patterned_sample = []
MAX = 500
iloc = 0
while(len(patterned_sample) < 20 and MAX > iloc):
  wafer = df_with_pattern.iloc[iloc]
  failureType = str(wafer["failureType"])
  if(CLASS_COUNTS[failureType] > 0):
    patterned_sample.append(wafer)
    CLASS_COUNTS[failureType] = CLASS_COUNTS[failureType] - 1
  iloc += 1


In [64]:
import pandas as pd
none_sample = df_without_pattern.sample(n=40, random_state=42)
mock_df = pd.concat([none_sample, pd.DataFrame(patterned_sample)], ignore_index=True)
mock_df = mock_df.sample(frac=1, random_state=42).reset_index(drop=True) 

In [65]:
# 載入模型
from explain import load_model

model_s1, metadata_s1 = load_model("../outputs/two_stage_20260531_110545/stage1")
model_s2, metadata_s2 = load_model("../outputs/two_stage_20260531_110545/stage2")

In [66]:
import torch

from models.inference import predict_two_stage, wafer_to_tensor

wafer = mock_df.iloc[0]["waferMap"]
image_size = metadata_s1["img_size"]

# 讓 tensor 與模型同裝置（mps/cuda/cpu）
device = next(model_s1.parameters()).device
wafer_tensor = wafer_to_tensor(wafer, image_size).to(device)
wafer_tensor = wafer_tensor.unsqueeze(0)
# inference

class_indices, class_names, scores = predict_two_stage(model_s1, model_s2, wafer_tensor)
print(class_names)
print(scores)

['none']
tensor([[1.3681e-08, 8.5027e-13, 4.6869e-07, 1.2849e-10, 2.2632e-10, 1.4958e-12,
         8.0781e-13, 1.1846e-10, 1.0000e+00]], device='mps:0')


In [ ]:
import cv2 

device = next(model_s1.parameters()).device
image_size = metadata_s1["img_size"]

# 1) 每張 wafer -> (3, H, W)，收集起來
tensors = [wafer_to_tensor(row["waferMap"], image_size) for _, row in mock_df.iterrows()]

# 2) 疊成一個 batch (B, 3, H, W) 並搬到模型裝置
batch = torch.stack(tensors).to(device)

# 3) 一次推論整批
class_indices, class_names, scores = predict_two_stage(model_s1, model_s2, batch)

# 4) 回填結果（scores 搬回 cpu 再轉 list 才能 json 序列化）
scores = scores.cpu()
records = []
for i, (_, row) in enumerate(mock_df.iterrows()):
  wafer = row["waferMap"]
  resized = cv2.resize(wafer, image_size, interpolation=cv2.INTER_NEAREST)
  records.append({
    "id": i+1,
    "height": int(resized.shape[0]),
    "width": int(resized.shape[1]),
    "wafer_map": wafer.tolist(),
    "pred_class": class_names[i],
    "pred_score": float(scores[i, class_indices[i]]),
  })

records

[{'id': 1,
  'height': 64,
  'width': 64,
  'wafer_map': [[0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0],
   [0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    1,
    2,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0],
   [0,
    0,
    0,
    0,
    0,
    0,
    2,
    1,
    1,
    2,
    1,
    2,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0],
   [0,
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    2,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    0],
   [0,
    0,
    0,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    0],
   [0,
    0,
    0,
    1,
  

In [68]:
import json
with open("wafer.json", "w") as f:
  json.dump(records, f)